In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
x_train = x_train / 255.0
x_test = x_test / 255.0

In [ ]:
model = models.Sequential([
    # Input layer + flatten 28x28 image
    layers.Flatten(input_shape=(28, 28)),
    # One hidden layer
    layers.Dense(128, activation='relu'),
    # Dropout
    layers.Dropout(0.2),
    # Output layer
    layers.Dense(10, activation='softmax')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    x_train,
    y_train,
    epochs=5,
    validation_split=0.1
)

Epoch 1/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9085 - loss: 0.3139 - val_accuracy: 0.9662 - val_loss: 0.1265
Epoch 2/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9552 - loss: 0.1518 - val_accuracy: 0.9757 - val_loss: 0.0887
Epoch 3/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9655 - loss: 0.1129 - val_accuracy: 0.9768 - val_loss: 0.0827
Epoch 4/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9714 - loss: 0.0935 - val_accuracy: 0.9785 - val_loss: 0.0759
Epoch 5/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9755 - loss: 0.0799 - val_accuracy: 0.9802 - val_loss: 0.0715


## Automated Search for Number of Neurons with Keras Tuner

In [ ]:
!pip install -q -U keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 2.0 MB/s eta 0:00:00


In [ ]:
import kerastuner as kt

/tmp/ipykernel_3027/1654478174.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


## Model Building:

In [ ]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

In [ ]:
def model_builder(hp):
    model = keras.Sequential()
    # Input layer
    model.add(keras.layers.Flatten(input_shape=(28, 28)))
    # Hyperparameter: number of neurons
    hp_units = hp.Int('units',min_value=16,max_value=512,step=16)
    # One hidden layer
    model.add(keras.layers.Dense(units=hp_units,activation='relu'))
    # Dropout
    model.add(keras.layers.Dropout(0.2))
    # Output layer
    model.add(keras.layers.Dense(10))
    # Compile model
    model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return model

In [ ]:
tuner = kt.Hyperband(
    model_builder,
    objective='val_accuracy',
    max_epochs=10,
    factor=3,
    directory='keras_tuner',
    project_name='mnist_ann')

In [ ]:
tuner.search(
    x_train,
    y_train,
    epochs=10,
    validation_split=0.1
)

Trial 30 Complete [00h 02m 15s]
val_accuracy: 0.24433332681655884

Best val_accuracy So Far: 0.5951666831970215
Total elapsed time: 00h 22m 28s


In [ ]:
tuner.results_summary()

Results summary
Results in keras_tuner/mnist_ann
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0021 summary
Hyperparameters:
units: 224
tuner/epochs: 4
tuner/initial_epoch: 0
tuner/bracket: 1
tuner/round: 0
Score: 0.5951666831970215

Trial 0026 summary
Hyperparameters:
units: 144
tuner/epochs: 10
tuner/initial_epoch: 0
tuner/bracket: 0
tuner/round: 0
Score: 0.46033334732055664

Trial 0019 summary
Hyperparameters:
units: 96
tuner/epochs: 4
tuner/initial_epoch: 0
tuner/bracket: 1
tuner/round: 0
Score: 0.4506666660308838

Trial 0009 summary
Hyperparameters:
units: 448
tuner/epochs: 2
tuner/initial_epoch: 0
tuner/bracket: 2
tuner/round: 0
Score: 0.43650001287460327

Trial 0027 summary
Hyperparameters:
units: 400
tuner/epochs: 10
tuner/initial_epoch: 0
tuner/bracket: 0
tuner/round: 0
Score: 0.414000004529953

Trial 0028 summary
Hyperparameters:
units: 80
tuner/epochs: 10
tuner/initial_epoch: 0
tuner/bracket: 0
tuner/round: 0
Score: 0.4138333201408386

Trial 0

### Combining Number of layers + Activation Functions + Learning rate

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from keras_tuner.tuners import RandomSearch
from tensorflow.keras.datasets import mnist
from tensorflow import keras

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

In [ ]:
def build_model(hp): # hyperparameters
    model = keras.Sequential()
    model.add(Flatten(input_shape=(28, 28)))

    hp_units1 = hp.Int('units1', min_value=32, max_value=512, step=32)
    hp_activ = hp.Choice('activ', ["relu", "sigmoid", "silu"])
    model.add(Dense(units=hp_units1, activation=hp_activ))

    hp_units2 = hp.Int('units2', min_value=32, max_value=512, step=32)
    model.add(Dense(units=hp_units2, activation=hp_activ))

    hp_learning_rate = hp.Choice('learning_rate', values=[0.01, 0.001, 0.0001])

    model.add(Dense(10, activation='softmax'))# dense layer

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model

In [ ]:
tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=3,
    directory='mnist_tuning',
    project_name='mnist'
)

In [ ]:
tuner.search(x_train, y_train, epochs=2, validation_data=(x_test, y_test))
tuner.results_summary()

Trial 3 Complete [00h 00m 23s]
val_accuracy: 0.9578999876976013

Best val_accuracy So Far: 0.9634000062942505
Total elapsed time: 00h 01m 11s
Results summary
Results in mnist_tuning/mnist
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0 summary
Hyperparameters:
units1: 32
activ: relu
units2: 320
learning_rate: 0.001
Score: 0.9634000062942505

Trial 2 summary
Hyperparameters:
units1: 128
activ: sigmoid
units2: 352
learning_rate: 0.01
Score: 0.9578999876976013

Trial 1 summary
Hyperparameters:
units1: 288
activ: sigmoid
units2: 192
learning_rate: 0.0001
Score: 0.9125999808311462


### Dimension Reduction

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
from keras.datasets import reuters
from keras.preprocessing import sequence

num_words = 1000

(reuters_train_x, reuters_train_y), (reuters_test_x, reuters_test_y) = \
    tf.keras.datasets.reuters.load_data(num_words=num_words)

n_labels = np.unique(reuters_train_y).shape[0]

2110848/2110848 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
n_labels

46

In [ ]:
reuters_train_x

array([list([1, 2, 2, 8, 43, 10, 447, 5, 25, 207, 270, 5, 2, 111, 16, 369, 186, 90, 67, 7, 89, 5, 19, 102, 6, 19, 124, 15, 90, 67, 84, 22, 482, 26, 7, 48, 4, 49, 8, 864, 39, 209, 154, 6, 151, 6, 83, 11, 15, 22, 155, 11, 15, 7, 48, 9, 2, 2, 504, 6, 258, 6, 272, 11, 15, 22, 134, 44, 11, 15, 16, 8, 197, 2, 90, 67, 52, 29, 209, 30, 32, 132, 6, 109, 15, 17, 12]),
       list([1, 2, 699, 2, 2, 56, 2, 2, 9, 56, 2, 2, 81, 5, 2, 57, 366, 737, 132, 20, 2, 7, 2, 49, 2, 2, 2, 2, 699, 2, 8, 7, 10, 241, 16, 855, 129, 231, 783, 5, 4, 587, 2, 2, 2, 775, 7, 48, 34, 191, 44, 35, 2, 505, 17, 12]),
       list([1, 53, 12, 284, 15, 14, 272, 26, 53, 959, 32, 818, 15, 14, 272, 26, 39, 684, 70, 11, 14, 12, 2, 18, 180, 183, 187, 70, 11, 14, 102, 32, 11, 29, 53, 44, 704, 15, 14, 19, 758, 15, 53, 959, 47, 2, 15, 14, 19, 132, 15, 39, 965, 32, 11, 14, 147, 72, 11, 180, 183, 187, 44, 11, 14, 102, 19, 11, 123, 186, 90, 67, 960, 4, 78, 13, 68, 467, 511, 110, 59, 89, 90, 67, 2, 55, 2, 92, 617, 80, 2, 46, 905, 220, 13,

In [ ]:
reuters_train_y

array([ 3,  4,  3, ..., 25,  3, 25])